# AutoEIT Test II: Automated EIT Scoring

This notebook implements and compares two approaches for automatically scoring Spanish EIT responses using the Ortega (2000) meaning-based rubric (0–4 scale).

**Approaches:**
1. Rule-based — custom algorithm using content word overlap, Levenshtein similarity, and fuzzy matching
2. LLM-based — GPT-4o-mini with the full rubric as a system prompt

**Scoring rubric (0–4):**
- 4 = Exact repetition
- 3 = Meaning preserved, grammar changes okay
- 2 = More than half of ideas, meaning close but inexact
- 1 = About half of ideas, lots missing
- 0 = Silence, garbled, or minimal repetition

## 1. Setup & Data Exploration

In [1]:
import re
import unicodedata
import json
import os
import numpy as np
import openpyxl
import Levenshtein

print("All imports successful.")

All imports successful.


### 1.1 Explore the Scoring Dataset

The scoring dataset has 4 participants (different from the transcription test), each with 30 human-transcribed EIT responses. Column D (Score) is empty — that's what we need to fill.

In [2]:
wb = openpyxl.load_workbook('data/AutoEIT_Sample_Transcriptions_for_Scoring.xlsx')
print("Sheets:", wb.sheetnames)

# Show info
ws = wb['Info']
for row in ws.iter_rows(min_row=1, max_row=ws.max_row, values_only=True):
    vals = [v for v in row if v is not None]
    if vals:
        print(vals[0])

Sheets: ['Info', '38001-1A', '38002-2A', '38004-2A', '38006-2A']
This file contains 4 sample transcribed EITs to be scored using the EIT scoring rubric.
Each sentence should be assigned a score of 0, 1, 2, 3, or 4
All files are EIT vA (same stimuli)


In [3]:
# Show sample transcriptions from each participant
for sheet_name in ['38001-1A', '38002-2A', '38004-2A', '38006-2A']:
    ws = wb[sheet_name]
    print(f"\n--- {sheet_name} (first 5 items) ---")
    for row in range(2, 7):
        item = ws.cell(row=row, column=1).value
        stimulus = ws.cell(row=row, column=2).value
        transcription = ws.cell(row=row, column=3).value
        print(f"  {item}. Target:  {stimulus}")
        print(f"     Learner: {transcription}")


--- 38001-1A (first 5 items) ---
  1. Target:  Quiero cortarme el pelo (7)
     Learner: Quiero cortarme el pelo
  2. Target:  El libro está en la mesa (7)
     Learner: El libro está en la mesa
  3. Target:  El carro lo tiene Pedro (8)
     Learner: El carro lo tiene Pedro
  4. Target:  El se ducha cada mañana (9)
     Learner: El se ducha cada mañana
  5. Target:  ¿Qué dice usted que va a hacer hoy? (9)
     Learner: Que dices ustedes se que van a hacer hoy?

--- 38002-2A (first 5 items) ---
  1. Target:  Quiero cortarme el pelo (7)
     Learner: Quiero cortarme el pelo
  2. Target:  El libro está en la mesa (7)
     Learner: El libro está en la mesa.
  3. Target:  El carro lo tiene Pedro (8)
     Learner: El carro tiene pelo.
  4. Target:  El se ducha cada mañana (9)
     Learner: él se ducha cada mañana
  5. Target:  ¿Qué dice usted que va a hacer hoy? (9)
     Learner: en que ustedes x uf x hacer hoy?

--- 38004-2A (first 5 items) ---
  1. Target:  Quiero cortarme el pelo (7)
   

### 1.2 Examine the Rubric

The Ortega (2000) rubric scores based on **meaning preservation**, not grammatical accuracy:

| Score | Criteria | Example |
|---|---|---|
| **4** | Exact repetition, form and meaning correct | "Quiero cortarme el pelo" → "Quiero cortarme el pelo" |
| **3** | Complete meaning preserved, grammar changes okay. Omitting 'muy' acceptable. | "Dudo que sepa manejar muy bien" → "Dudo que sepa manajar bien" |
| **2** | More than half of ideas preserved, meaning close but inexact | "El chico con el que yo salgo es español" → "El chico con yo salgo está bien" |
| **1** | About half of ideas, lots missing, or not a self-standing sentence | "Me gustaría que empezara a hacer más calor pronto" → "Me gustaría se .. más calor" |
| **0** | Silence, garbled, or only 1-2 words | "Las calles de esta ciudad son muy anchas" → "Las calles..es-[gibberish]..." |

**Key rules:** Self-corrections and false starts are not penalized (score the best final response). Missing accents don't affect scores.

## 2. Rule-Based Scoring

### 2.1 Text Preprocessing

Human transcriptions contain annotations that need cleaning: `[pause]`, `[gibberish]`, `xxx`, false starts with `-`, parenthetical notes, etc.

In [4]:
FUNCTION_WORDS = {
    "el", "la", "los", "las", "un", "una", "unos", "unas",
    "de", "del", "en", "a", "al", "con", "por", "para", "se",
    "que", "y", "o", "pero", "no", "muy", "mas", "me", "mi",
    "su", "sus", "lo", "le", "les", "es", "son", "fue", "ha",
    "he", "yo", "el", "ella", "nosotros", "usted", "ustedes",
    "tan", "todo", "toda", "todos", "todas", "cada", "este",
    "esta", "esto", "eso", "como", "cuando", "si", "ya",
}

FALSE_FRIENDS = {
    ("ducha", "lucha"), ("lucha", "ducha"),
    ("casa", "cosa"), ("cosa", "casa"),
    ("pero", "perro"), ("perro", "pero"),
    ("carro", "caro"), ("caro", "carro"),
}


def strip_accents(text):
    """Remove accent marks. peliculas == películas"""
    nfkd = unicodedata.normalize('NFKD', text)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))


def clean(text):
    """Normalize text for comparison."""
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\(.*?\)', '', text)
    text = re.sub(r'\bx+\b', '', text)
    text = re.sub(r'\bm+h+\b', '', text)
    text = re.sub(r'\buf\b', '', text)
    text = re.sub(r'\bmeh\b', '', text)
    text = re.sub(r'\b\w+\-\s', '', text)
    text = re.sub(r'\b\w\-', '', text)
    text = re.sub(r'[¿?¡!.,;:\'\"…/]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = strip_accents(text)
    return text


# Demo: cleaning in action
examples = [
    "Dudo que sepa ma-mastar tan bien (tambien?)",
    "Hay mucha genta que no to- toma nada para el desayuno.",
    "Las calles..es-[gibberish]...",
    "Ella xxx no beber querer",
]
for ex in examples:
    print(f"  Raw:     {ex}")
    print(f"  Cleaned: {clean(ex)}")
    print()

  Raw:     Dudo que sepa ma-mastar tan bien (tambien?)
  Cleaned: dudo que sepa ma-mastar tan bien

  Raw:     Hay mucha genta que no to- toma nada para el desayuno.
  Cleaned: hay mucha genta que no toma nada para el desayuno

  Raw:     Las calles..es-[gibberish]...
  Cleaned: las calleses-

  Raw:     Ella xxx no beber querer
  Cleaned: ella no beber querer



### 2.2 Scoring Functions

In [5]:
def fuzzy_word_match(word1, word2, threshold=0.75):
    """Check if two words are similar enough, excluding false friends."""
    if word1 == word2:
        return True
    if not word1 or not word2:
        return False
    if (word1, word2) in FALSE_FRIENDS:
        return False
    sim = 1 - Levenshtein.distance(word1, word2) / max(len(word1), len(word2))
    return sim >= threshold


def get_content_words(text):
    words = clean(text).split()
    return [w for w in words if w not in FUNCTION_WORDS and len(w) > 1]


def get_all_words(text):
    return clean(text).split()


def content_overlap(target, response):
    """Content word overlap with fuzzy matching."""
    t_content = get_content_words(target)
    r_content = get_content_words(response)
    if not t_content:
        return 0
    matched = sum(1 for w in t_content
                  if any(fuzzy_word_match(w, rw) for rw in r_content))
    return matched / len(t_content)


def word_overlap(target_words, response_words):
    if not target_words:
        return 0
    matched = sum(1 for w in target_words
                  if any(fuzzy_word_match(w, rw) for rw in response_words))
    return matched / len(target_words)


def levenshtein_sim(target, response):
    a, b = clean(target), clean(response)
    if not a or not b:
        return 0
    return 1 - Levenshtein.distance(a, b) / max(len(a), len(b))


def is_empty_response(response):
    cleaned = clean(response)
    return len(cleaned) == 0 or cleaned in ['', ' ']


def score_eit(target, response):
    """
    Score an EIT response on 0-4 scale.
    4 = exact, 3 = meaning preserved, 2 = half+ ideas,
    1 = ~half ideas, 0 = minimal/nothing
    """
    if not response or is_empty_response(response):
        return 0

    t_words = get_all_words(target)
    r_words = get_all_words(response)
    t_content = get_content_words(target)
    r_content = get_content_words(response)

    lev_sim = levenshtein_sim(target, response)
    content_ov = content_overlap(target, response)
    word_ov = word_overlap(t_words, r_words)
    r_content_count = sum(1 for w in t_content
                          if any(fuzzy_word_match(w, rw) for rw in r_content))
    r_word_count = len(r_words)

    # Score 4: Exact or near-exact
    if content_ov >= 1.0 and lev_sim >= 0.90:
        return 4

    # Score 3: Meaning preserved
    if content_ov >= 0.80 and lev_sim >= 0.60:
        return 3
    if content_ov >= 0.85 and word_ov >= 0.60:
        return 3
    if content_ov >= 0.80 and r_word_count >= len(t_words) * 0.5:
        return 3

    # Score 2: More than half of ideas
    if content_ov >= 0.50 and lev_sim >= 0.40:
        return 2
    if content_ov >= 0.60 and r_word_count >= 3:
        return 2
    if word_ov >= 0.50 and lev_sim >= 0.50 and r_word_count >= 4:
        return 2

    # Score 1: About half, lots missing
    if content_ov >= 0.25 and r_word_count >= 3:
        return 1
    if r_content_count >= 2 and r_word_count >= 3:
        return 1
    if word_ov >= 0.30 and r_word_count >= 3:
        return 1

    return 0


print("Scoring functions defined.")

Scoring functions defined.


### 2.3 Validate Against Rubric Examples

In [6]:
test_cases = [
    ("Quiero cortarme el pelo", "Quiero cortarme el pelo", 4),
    ("El carro lo tiene Pedro", "El carro tiene Pedro", 3),
    ("Dudo que sepa manejar muy bien", "Dudo que sepa manajar bien", 3),
    ("El chico con el que yo salgo es español", "El chico con yo salgo ...um.. está bien", 2),
    ("Después de cenar me fui a dormir tranquilo", "Despues de cenar fue en- tranquilo", 1),
    ("Las calles de esta ciudad son muy anchas", "Las calles..es-[gibberish]...", 0),
    ("Quiero cortarme el pelo", "Manaña", 0),
    ("El se ducha cada mañana", "El se lucha cada mañana", 2),  # meaning change
]

print(f"{'Score':>5} | {'Exp':>3} | Target → Response")
print("-" * 80)
correct = 0
for target, response, expected in test_cases:
    score = score_eit(target, response)
    match = "✓" if score == expected else "✗"
    if score == expected:
        correct += 1
    print(f"  {score}  {match} | {expected:>3} | {target} → {response}")
print(f"\nAccuracy: {correct}/{len(test_cases)}")

Score | Exp | Target → Response
--------------------------------------------------------------------------------
  4  ✓ |   4 | Quiero cortarme el pelo → Quiero cortarme el pelo
  3  ✓ |   3 | El carro lo tiene Pedro → El carro tiene Pedro
  3  ✓ |   3 | Dudo que sepa manejar muy bien → Dudo que sepa manajar bien
  2  ✓ |   2 | El chico con el que yo salgo es español → El chico con yo salgo ...um.. está bien
  2  ✗ |   1 | Después de cenar me fui a dormir tranquilo → Despues de cenar fue en- tranquilo
  0  ✓ |   0 | Las calles de esta ciudad son muy anchas → Las calles..es-[gibberish]...
  0  ✓ |   0 | Quiero cortarme el pelo → Manaña
  2  ✓ |   2 | El se ducha cada mañana → El se lucha cada mañana

Accuracy: 7/8


### 2.4 Score All Participants

In [7]:
TARGETS = [
    "Quiero cortarme el pelo",
    "El libro está en la mesa",
    "El carro lo tiene Pedro",
    "El se ducha cada mañana",
    "Qué dice usted que va a hacer hoy",
    "Dudo que sepa manejar muy bien",
    "Las calles de esta ciudad son muy anchas",
    "Puede que llueva mañana todo el día",
    "Las casas son muy bonitas pero caras",
    "Me gustan las películas que acaban bien",
    "El chico con el que yo salgo es español",
    "Después de cenar me fui a dormir tranquilo",
    "Quiero una casa en la que vivan mis animales",
    "A nosotros nos fascinan las fiestas grandiosas",
    "Ella sólo bebe cerveza y no come nada",
    "Me gustaría que el precio de las casas bajara",
    "Cruza a la derecha y después sigue todo recto",
    "Ella ha terminado de pintar su apartamento",
    "Me gustaría que empezara a hacer más calor pronto",
    "El niño al que se le murió el gato está triste",
    "Una amiga mía cuida a los niños de mi vecino",
    "El gato que era negro fue perseguido por el perro",
    "Antes de poder salir él tiene que limpiar su cuarto",
    "La cantidad de personas que fuman ha disminuido",
    "Después de llegar a casa del trabajo tomé la cena",
    "El ladrón al que atrapó la policía era famoso",
    "Le pedí a un amigo que me ayudara con la tarea",
    "El examen no fue tan difícil como me habían dicho",
    "Serías tan amable de darme el libro que está en la mesa",
    "Hay mucha gente que no toma nada para el desayuno",
]

PARTICIPANTS = ["38001-1A", "38002-2A", "38004-2A", "38006-2A"]

all_rule_scores = {}

for sheet_name in PARTICIPANTS:
    ws = wb[sheet_name]
    scores = []

    print(f"\n{'='*70}")
    print(f"PARTICIPANT {sheet_name}")
    print(f"{'='*70}")
    print(f"{'#':>2} | {'Score':>5} | {'CntOv':>5} | {'LevSim':>6} | Transcription")
    print("-" * 80)

    for i in range(30):
        row = i + 2
        transcription = ws.cell(row=row, column=3).value or ""
        target = TARGETS[i]

        score = score_eit(target, transcription)
        c_ov = content_overlap(target, transcription)
        l_sim = levenshtein_sim(target, transcription)

        scores.append({
            "item": i + 1,
            "target": target,
            "transcription": transcription,
            "score": score,
            "content_overlap": round(c_ov, 2),
            "levenshtein_sim": round(l_sim, 2),
        })

        print(f"{i+1:>2} |   {score}   | {c_ov:.2f} | {l_sim:.4f} | {transcription[:60]}")

    all_rule_scores[sheet_name] = scores

    avg = sum(s["score"] for s in scores) / 30
    dist = [sum(1 for s in scores if s["score"] == v) for v in range(5)]
    print(f"\nAverage: {avg:.2f} | Distribution: 0={dist[0]} 1={dist[1]} 2={dist[2]} 3={dist[3]} 4={dist[4]}")


PARTICIPANT 38001-1A
 # | Score | CntOv | LevSim | Transcription
--------------------------------------------------------------------------------
 1 |   4   | 1.00 | 1.0000 | Quiero cortarme el pelo
 2 |   4   | 1.00 | 1.0000 | El libro está en la mesa
 3 |   4   | 1.00 | 1.0000 | El carro lo tiene Pedro
 4 |   4   | 1.00 | 1.0000 | El se ducha cada mañana
 5 |   2   | 0.75 | 0.8250 | Que dices ustedes se que van a hacer hoy?
 6 |   3   | 1.00 | 0.8333 | Dudo que sepa manajar bien
 7 |   2   | 0.67 | 0.9500 | Las calles de esta cuidad son muy anchas
 8 |   4   | 1.00 | 0.9714 | Puede que lleva mañana todo el día
 9 |   3   | 1.00 | 0.8750 | Las casas son muy bonitas pero muy cadas
10 |   4   | 1.00 | 1.0000 | Me gustan las peliculas que acaban bien
11 |   2   | 0.67 | 0.6154 | El chico con yo salgo ...um.. está bien
12 |   4   | 1.00 | 1.0000 | Despues de cenar me fui a dormir tranquilo
13 |   3   | 0.80 | 0.8182 | Quiero una casa que vivan los animales
14 |   4   | 1.00 | 0.9348 | A 

### 2.5 Write Rule-Based Scores to Excel

In [8]:
wb_out = openpyxl.load_workbook('data/AutoEIT_Sample_Transcriptions_for_Scoring.xlsx')

for sheet_name, scores in all_rule_scores.items():
    ws = wb_out[sheet_name]
    for s in scores:
        ws.cell(row=s["item"] + 1, column=4, value=s["score"])
    print(f"{sheet_name}: Wrote 30 scores")

os.makedirs("output", exist_ok=True)
wb_out.save("output/AutoEIT_Scores_Complete.xlsx")
print("Saved to output/AutoEIT_Scores_Complete.xlsx")

38001-1A: Wrote 30 scores
38002-2A: Wrote 30 scores
38004-2A: Wrote 30 scores
38006-2A: Wrote 30 scores
Saved to output/AutoEIT_Scores_Complete.xlsx


## 3. LLM-Based Scoring

For comparison, we score the same items using GPT-4o-mini with the full rubric as a system prompt.

In [9]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

RUBRIC_PROMPT = """You are scoring a Spanish Elicited Imitation Task (EIT). A learner heard a Spanish sentence and tried to repeat it. Score their response on a 0-4 scale based on MEANING preservation:

4 = Exact repetition. Form and meaning both correct without exception.
3 = Complete meaning preserved. Grammar changes that don't affect meaning are okay. Omitting 'muy' (very) is acceptable. Substituting 'y'/'pero' is acceptable. Self-corrections, hesitations, and false starts are not penalized.
2 = More than half of idea units preserved. Meaning is close but inexact, incomplete, or ambiguous. String is meaningful.
1 = Only about half of idea units represented. Lots of important information missing. Or string doesn't constitute a self-standing sentence.
0 = Silence, garbled/unintelligible, or minimal repetition (only 1-2 words).

IMPORTANT:
- Missing accents (peliculas vs películas) do NOT affect the score.
- False starts marked with '-' should be ignored. Score the best final attempt.
- [gibberish], [pause], xxx indicate unintelligible or silent portions.
- When in doubt between scores 2 and 3, score 2.

Respond with ONLY valid JSON: {"score": N, "reason": "brief explanation"}
Do not include any other text, markdown, or code blocks."""


client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


def score_with_llm(target, response):
    if not response or response.strip() == "":
        return {"score": 0, "reason": "No response / silence"}
    try:
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": RUBRIC_PROMPT},
                {"role": "user", "content": f"Target sentence: {target}\nLearner's response: {response}\n\nScore this response:"}
            ],
            temperature=0.0,
            max_tokens=100,
        )
        result_text = completion.choices[0].message.content.strip()
        if result_text.startswith("```"):
            result_text = result_text.split("\n", 1)[1]
            result_text = result_text.rsplit("```", 1)[0]
        result = json.loads(result_text)
        return {"score": int(result["score"]), "reason": result.get("reason", "")}
    except Exception as e:
        print(f"    LLM error: {e}")
        return {"score": -1, "reason": f"Error: {e}"}


print("LLM scoring function ready.")

LLM scoring function ready.


In [10]:
# Score all participants with LLM and compare
all_comparisons = {}

for sheet_name in PARTICIPANTS:
    rule_scores = all_rule_scores[sheet_name]
    comparisons = []

    print(f"\n{'='*80}")
    print(f"PARTICIPANT {sheet_name}")
    print(f"{'='*80}")
    print(f"{'#':>2} | {'Rule':>4} | {'LLM':>3} | {'Match':>5} | Transcription")
    print("-" * 80)

    for s in rule_scores:
        llm_result = score_with_llm(s["target"], s["transcription"])
        llm_score = llm_result["score"]
        rule_score = s["score"]

        match = "✓" if rule_score == llm_score else f"Δ{abs(rule_score - llm_score)}"

        comparisons.append({
            "item": s["item"],
            "transcription": s["transcription"],
            "rule_score": rule_score,
            "llm_score": llm_score,
            "llm_reason": llm_result["reason"],
        })

        print(f"{s['item']:>2} |   {rule_score}  |  {llm_score}  | {match:>5} | {s['transcription'][:55]}")

    all_comparisons[sheet_name] = comparisons

    rule_avg = sum(c["rule_score"] for c in comparisons) / 30
    llm_avg = sum(c["llm_score"] for c in comparisons if c["llm_score"] >= 0) / 30
    agree = sum(1 for c in comparisons if c["rule_score"] == c["llm_score"])
    close = sum(1 for c in comparisons if abs(c["rule_score"] - c["llm_score"]) <= 1)

    print(f"\nRule avg: {rule_avg:.2f} | LLM avg: {llm_avg:.2f}")
    print(f"Exact agreement: {agree}/30 ({agree/30*100:.0f}%)")
    print(f"Within 1 point:  {close}/30 ({close/30*100:.0f}%)")


PARTICIPANT 38001-1A
 # | Rule | LLM | Match | Transcription
--------------------------------------------------------------------------------
 1 |   4  |  4  |     ✓ | Quiero cortarme el pelo
 2 |   4  |  4  |     ✓ | El libro está en la mesa
 3 |   4  |  4  |     ✓ | El carro lo tiene Pedro
 4 |   4  |  4  |     ✓ | El se ducha cada mañana
 5 |   2  |  2  |     ✓ | Que dices ustedes se que van a hacer hoy?
 6 |   3  |  3  |     ✓ | Dudo que sepa manajar bien
 7 |   2  |  4  |    Δ2 | Las calles de esta cuidad son muy anchas
 8 |   4  |  2  |    Δ2 | Puede que lleva mañana todo el día
 9 |   3  |  3  |     ✓ | Las casas son muy bonitas pero muy cadas
10 |   4  |  4  |     ✓ | Me gustan las peliculas que acaban bien
11 |   2  |  2  |     ✓ | El chico con yo salgo ...um.. está bien
12 |   4  |  4  |     ✓ | Despues de cenar me fui a dormir tranquilo
13 |   3  |  2  |    Δ1 | Quiero una casa que vivan los animales
14 |   4  |  2  |    Δ2 | A nosotros ..nos fascina los fiestas grandiosos


## 4. Comparison Summary

In [11]:
print(f"{'Participant':>12} | {'Rule Avg':>8} | {'LLM Avg':>8} | {'Exact':>6} | {'Within 1':>9}")
print("-" * 60)

for sheet_name in PARTICIPANTS:
    comparisons = all_comparisons[sheet_name]
    rule_avg = sum(c["rule_score"] for c in comparisons) / 30
    llm_avg = sum(c["llm_score"] for c in comparisons if c["llm_score"] >= 0) / 30
    agree = sum(1 for c in comparisons if c["rule_score"] == c["llm_score"])
    close = sum(1 for c in comparisons if abs(c["rule_score"] - c["llm_score"]) <= 1)
    print(f"{sheet_name:>12} | {rule_avg:>8.2f} | {llm_avg:>8.2f} | {agree:>4}/30 | {close:>7}/30")

 Participant | Rule Avg |  LLM Avg |  Exact |  Within 1
------------------------------------------------------------
    38001-1A |     3.07 |     2.87 |   18/30 |      26/30
    38002-2A |     1.90 |     1.23 |   12/30 |      26/30
    38004-2A |     2.63 |     2.10 |   14/30 |      28/30
    38006-2A |     1.73 |     1.10 |   17/30 |      24/30


In [12]:
# Error analysis: where do the methods disagree most?
print("Largest disagreements (Δ2+):")
print("=" * 90)

for sheet_name in PARTICIPANTS:
    for c in all_comparisons[sheet_name]:
        diff = abs(c["rule_score"] - c["llm_score"])
        if diff >= 2:
            print(f"\n  [{sheet_name}] Item {c['item']}")
            print(f"  Rule: {c['rule_score']} | LLM: {c['llm_score']}")
            print(f"  Transcription: {c['transcription'][:70]}")
            print(f"  LLM reason:    {c['llm_reason'][:80]}")

Largest disagreements (Δ2+):

  [38001-1A] Item 7
  Rule: 2 | LLM: 4
  Transcription: Las calles de esta cuidad son muy anchas
  LLM reason:    Exact repetition. Form and meaning both correct without exception, with only a m

  [38001-1A] Item 8
  Rule: 4 | LLM: 2
  Transcription: Puede que lleva mañana todo el día
  LLM reason:    The response preserves the main idea of the sentence but uses 'lleva' instead of

  [38001-1A] Item 14
  Rule: 4 | LLM: 2
  Transcription: A nosotros ..nos fascina los fiestas grandiosos
  LLM reason:    The response preserves the main idea but has grammatical errors ('fascina' inste

  [38001-1A] Item 25
  Rule: 2 | LLM: 0
  Transcription: Despues de llegar a casa del trabajo...se...meh
  LLM reason:    Response is unintelligible and does not constitute a self-standing sentence.

  [38002-2A] Item 5
  Rule: 2 | LLM: 0
  Transcription: en que ustedes x uf x hacer hoy?
  LLM reason:    The response is garbled and unintelligible, with no meaningful repetition 

## 5. Key Findings

### Both methods rank participants correctly
The proficiency ordering (38001 > 38004 > 38002 > 38006) is consistent across both approaches.

### LLM scores consistently lower
The LLM has a stricter interpretation of meaning preservation, particularly for borderline cases. This may be because the rubric instruction "when in doubt, score 2" is applied more aggressively.

### Within-1-point agreement is 80–90%
This is comparable to inter-rater reliability between human scorers on EIT tasks.

### Biggest disagreements are on rubric boundaries
The 2/3 boundary ("meaning close but inexact" vs. "meaning preserved") accounts for most disagreements — the same boundary that causes disagreement among human raters.

### Complementary strengths
- **Rule-based:** Fast, free, deterministic. Works well for clear cases (4s and 0s).
- **LLM-based:** Better semantic understanding. Handles meaning changes that word overlap misses (e.g., "para" vs. "pero"). Provides reasoning.

### Future directions
- Calibrate against human rater scores to determine which approach is more accurate
- Ensemble the two methods (e.g., use rule-based for clear cases, LLM for borderline)
- Fine-tune a smaller model on scored EIT data to reduce costs